# 01. Xây dựng Digital Burnout Indicator Framework (DBI)

Notebook này thực hiện quá trình tích hợp các thành phần đã được xây dựng từ hai nhánh dữ liệu quốc tế và Việt Nam nhằm hoàn thiện Digital Burnout Indicator Framework (DBI).

Framework DBI được xây dựng dựa trên ba nguồn bằng chứng chính:

1. Cơ sở lý thuyết về Digital Burnout, Technostress, Cognitive Overload và Recovery Theory.
2. Kết quả phân tích dữ liệu quốc tế thông qua Feature Validation và Explainable Machine Learning.
3. Khả năng đo lường thực tế trên dữ liệu khảo sát sinh viên Việt Nam.

Mục tiêu của notebook:

- Tổng hợp các nhóm dimension của Digital Burnout.
- Chuẩn hóa danh sách candidate indicators.
- Liên kết indicator lý thuyết với feature khảo sát sinh viên Việt Nam.
- Bổ sung bằng chứng thực nghiệm từ dataset quốc tế.
- Xây dựng bộ DBI Framework hoàn chỉnh phục vụ đánh giá và so sánh giữa các dataset.

Output cuối cùng của notebook là bộ framework DBI bao gồm:

- Dimension.
- Indicator.
- Feature mapping.
- Measurement direction.
- Evidence support.

# 0. Set Up

Thiết lập môi trường phân tích và khai báo các đường dẫn dữ liệu cần thiết cho quá trình xây dựng DBI Framework.

In [205]:
# Import các thư viện cần thiết

import pandas as pd
import numpy as np

from pathlib import Path

print("Đã tải các thư viện cần thiết.")

Đã tải các thư viện cần thiết.


In [206]:
# Thiết lập đường dẫn dữ liệu đầu vào

international_directory = Path(
    "../../data/processed/international_dataset"
)

vietnam_directory = Path(
    "../../data/processed/vietnam_dataset"
)

output_directory = Path(
    "../../data/processed/cross_dataset_analysis"
)

output_directory.mkdir(
    parents=True,
    exist_ok=True
)

print("Đã thiết lập đường dẫn dữ liệu.")

Đã thiết lập đường dẫn dữ liệu.


# 1. Load DBI Components

Tải các thành phần DBI đã được xây dựng từ hai pipeline dữ liệu quốc tế và Việt Nam.

Các thành phần được sử dụng bao gồm:

*Vietnam Dataset*

- Candidate indicators: Danh sách indicator dựa trên cơ sở lý thuyết.

- Feature-indicator mapping: Liên kết indicator với feature khảo sát sinh viên Việt Nam.

- Validated features: Các feature đạt yêu cầu sau quá trình Feature Validation.

*International Dataset*

- Feature validation: Bằng chứng thống kê về mức độ liên quan của feature.

- SHAP importance: Bằng chứng từ mô hình Machine Learning về mức độ đóng góp của feature.

In [207]:
# Đọc các thành phần DBI từ Vietnam Dataset

candidate_indicators = pd.read_csv(
    vietnam_directory / "candidate_indicators.csv"
)

feature_indicator_mapping = pd.read_csv(
    vietnam_directory / "feature_indicator_mapping.csv"
)

vietnam_validated_features = pd.read_csv(
    vietnam_directory / "validated_features.csv"
)

print("Đã tải dữ liệu thành phần từ Vietnam Dataset.")

Đã tải dữ liệu thành phần từ Vietnam Dataset.


In [208]:
# Đọc các thành phần bằng chứng từ International Dataset

feature_validation_summary = pd.read_csv(
    international_directory / "feature_validation_summary.csv"
)

shap_feature_importance = pd.read_csv(
    international_directory / "shap_feature_importance.csv"
)

international_validated_features = pd.read_csv(
    international_directory / "validated_features.csv"
)

print("Đã tải dữ liệu thành phần từ International Dataset.")

Đã tải dữ liệu thành phần từ International Dataset.


In [209]:
# Kiểm tra kích thước các dataframe

print(
    f"Số lượng candidate indicators: {candidate_indicators.shape[0]}"
)

print(
    f"Số lượng feature mapping: {feature_indicator_mapping.shape[0]}"
)

print(
    f"Số lượng SHAP features: {shap_feature_importance.shape[0]}"
)

Số lượng candidate indicators: 23
Số lượng feature mapping: 17
Số lượng SHAP features: 26


# 2. DBI Structural Framework

Xây dựng cấu trúc các nhóm chỉ số (DBI Dimensions) trong Digital Burnout Indicator Framework dựa trên hệ thống candidate indicators đã được xác định từ quá trình tổng quan tài liệu và thiết kế bộ khảo sát sinh viên Việt Nam.

Digital Burnout được xem là hiện tượng đa chiều, chịu ảnh hưởng đồng thời bởi hành vi sử dụng công nghệ, trạng thái tâm lý, hiệu suất nhận thức và khả năng phục hồi.

Do đó, toàn bộ candidate indicators sẽ được tổ chức thành bốn nhóm chỉ số chính (DBI Dimensions), bao gồm:

- Digital Exposure
- Psychological Symptoms
- Cognitive Performance
- Sleep & Recovery

Việc xây dựng cấu trúc này giúp chuẩn hóa hệ thống chỉ số trước khi tích hợp bằng chứng thực nghiệm từ bộ dữ liệu quốc tế ở các bước tiếp theo.

In [210]:
# Trích xuất danh sách các dimension trong DBI Framework

dbi_dimension_framework = (
    candidate_indicators[
        [
            "dbi_dimension"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        by="dbi_dimension"
    )
    .reset_index(
        drop=True
    )
)

In [211]:
# Bổ sung mã định danh cho từng dimension

dimension_id_mapping = {

    "Digital Exposure": "DE",
    "Psychological Symptoms": "PS",
    "Cognitive Performance": "CP",
    "Sleep & Recovery": "SR"

}


dbi_dimension_framework[
    "dimension_id"
] = (
    dbi_dimension_framework[
        "dbi_dimension"
    ]
    .map(
        dimension_id_mapping
    )
)

In [212]:
# Bổ sung mô tả cho từng dimension

dimension_definition_mapping = {

    "Digital Exposure":
        "Phản ánh mức độ tiếp xúc và hành vi sử dụng thiết bị số.",
    "Psychological Symptoms":
        "Phản ánh các biểu hiện về căng thẳng, kiệt sức và ảnh hưởng tâm lý do công nghệ số.",
    "Cognitive Performance":
        "Phản ánh sự thay đổi về khả năng tập trung, xử lý thông tin và hiệu suất nhận thức.",
    "Sleep & Recovery":
        "Phản ánh khả năng phục hồi thông qua giấc ngủ và thời gian nghỉ ngơi."

}

dbi_dimension_framework[
    "dimension_definition"
] = (
    dbi_dimension_framework[
        "dbi_dimension"
    ]
    .map(
        dimension_definition_mapping
    )
)

In [213]:
# Sắp xếp lại thứ tự các cột

dbi_dimension_framework = (
    dbi_dimension_framework[
        [
            "dimension_id",
            "dbi_dimension",
            "dimension_definition"
        ]
    ]
)

display(
    dbi_dimension_framework
)

,dimension_id,dbi_dimension,dimension_definition
0,CP,Cognitive Performance,"Phản ánh sự thay đổi về khả năng tập trung, xử..."
1,DE,Digital Exposure,Phản ánh mức độ tiếp xúc và hành vi sử dụng th...
2,PS,Psychological Symptoms,"Phản ánh các biểu hiện về căng thẳng, kiệt sức..."
3,SR,Sleep & Recovery,Phản ánh khả năng phục hồi thông qua giấc ngủ ...


In [214]:
# Thống kê số lượng indicator thuộc từng dimension

dimension_summary = (
    candidate_indicators
    .groupby(
        "dbi_dimension"
    )
    .agg(
        indicator_count=(
            "indicator_id",
            "count"
        )
    )
    .reset_index()
)

display(
    dimension_summary
)

,dbi_dimension,indicator_count
0,Cognitive Performance,4
1,Digital Exposure,6
2,Psychological Symptoms,8
3,Sleep & Recovery,5


In [215]:
# Lưu DBI Dimension Framework

dimension_output_path = (
    output_directory /
    "dbi_dimension_framework.csv"
)

dbi_dimension_framework.to_csv(
    dimension_output_path,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Đã lưu DBI Dimension Framework."
)

Đã lưu DBI Dimension Framework.


# 3. DBI Indicator Evidence Matrix

Xây dựng ma trận bằng chứng (DBI Indicator Evidence Matrix) nhằm tổng hợp mức độ hỗ trợ của từng Digital Burnout Indicator từ cả góc độ lý thuyết và thực nghiệm.

Mỗi indicator trong DBI Framework sẽ được đánh giá dựa trên ba nguồn bằng chứng chính:

- **Theoretical Evidence:** Cơ sở lý thuyết của indicator được kế thừa từ quá trình tổng quan tài liệu và xây dựng Candidate Indicators.

- **Vietnam Measurement Evidence:** Khả năng đo lường indicator thông qua các biến quan sát trong bộ khảo sát sinh viên Việt Nam.

- **International Empirical Evidence:** Bằng chứng thực nghiệm thu được từ quá trình Feature Validation và Explainable AI trên bộ dữ liệu Digital Burnout quốc tế.

Việc tổng hợp các nguồn bằng chứng này giúp xác định mức độ tin cậy của từng indicator trước khi xây dựng Digital Burnout Indicator Framework hoàn chỉnh.

## 3.1. Theoretical Evidence

Xác định bằng chứng lý thuyết của từng Digital Burnout Indicator.

Toàn bộ indicator trong nghiên cứu đều được xây dựng dựa trên các mô hình và nghiên cứu trước đây về Digital Burnout, Technostress, Media Multitasking, Cognitive Overload và Recovery Theory.

Do đó, tất cả các indicator đều được xem là có nền tảng lý thuyết đầy đủ.

In [216]:
# Trích xuất thông tin lý thuyết của từng indicator

theoretical_evidence = (
    candidate_indicators[
        [
            "indicator_id",
            "indicator_name",
            "dbi_dimension",
            "theoretical_support"
        ]
    ]
    .copy()
)

theoretical_evidence[
    "theory_supported"
] = True


display(
    theoretical_evidence.head()
)

,indicator_id,indicator_name,dbi_dimension,theoretical_support,theory_supported
0,DE01,Daily Total Screen Time,Digital Exposure,Technostress Creators Framework (Techno-overload),True
1,DE02,Media Multitasking Frequency,Digital Exposure,Media Multitasking Index (MMI),True
2,DE03,After-hours Work Connectivity,Digital Exposure,Workplace Telepressure Theory,True
3,DE04,Notification Check Frequency,Digital Exposure,Technostress Creators - Techno-invasion,True
4,DE05,Late-night Screen Exposure,Digital Exposure,Circadian Phase-Shift Theory,True


## 3.2 Vietnam Measurement Evidence

Đánh giá khả năng đo lường từng Digital Burnout Indicator trên bộ dữ liệu khảo sát sinh viên Việt Nam.

Thông qua bảng Feature-Indicator Mapping, mỗi indicator được liên kết với một hoặc nhiều biến quan sát trong bộ khảo sát.

Nếu indicator đã được ánh xạ với ít nhất một feature khảo sát thì indicator được xem là có khả năng đo lường trong bối cảnh sinh viên Việt Nam.

In [217]:
# Thống kê số lượng feature đại diện cho từng indicator

measurement_evidence = (
    feature_indicator_mapping
    .groupby(
        [
            "indicator_id",
            "indicator_name",
            "dbi_dimension"
        ]
    )
    .agg(
        mapped_feature_count=(
            "feature",
            "count"
        )
    )
    .reset_index()
)

measurement_evidence[
    "vietnam_measurable"
] = (
    measurement_evidence[
        "mapped_feature_count"
    ] > 0
)

display(
    measurement_evidence.head()
)

,indicator_id,indicator_name,dbi_dimension,mapped_feature_count,vietnam_measurable
0,CP02,Techno-Complexity/Cognitive Overload,Cognitive Performance,1,True
1,CP03,Sustained Attention Decline,Cognitive Performance,3,True
2,CP04,Cross-Domain Interference Susceptibility,Cognitive Performance,1,True
3,DE01,Daily Total Screen Time,Digital Exposure,1,True
4,DE02,Media Multitasking Frequency,Digital Exposure,2,True


## 3.3 International Empirical Evidence

Đánh giá mức độ hỗ trợ thực nghiệm của từng feature dựa trên kết quả Feature Validation của bộ dữ liệu Digital Burnout quốc tế.

Kết quả Feature Validation được sử dụng để phân loại các feature thành ba nhóm:

- Core Feature
- Validated Feature
- Supporting Feature

Việc phân loại này phản ánh mức độ quan trọng của từng feature trong quá trình dự đoán Digital Burnout trên bộ dữ liệu quốc tế.

In [218]:
# Trích xuất kết quả Feature Validation

validation_evidence = (
    international_validated_features[
        [
            "feature",
            "feature_status"
        ]
    ]
    .copy()
)

display(
    validation_evidence.head()
)

,feature,feature_status
0,emotional_exhaustion,Core Feature
1,stress_level,Core Feature
2,daily_screen_time,Core Feature
3,work_satisfaction,Core Feature
4,doomscrolling_duration,Core Feature


## 3.4 Explainable AI Evidence

Bổ sung bằng chứng Explainable AI cho từng feature thông qua kết quả phân tích SHAP trên mô hình XGBoost.

Giá trị SHAP phản ánh mức độ đóng góp của từng feature vào quyết định dự đoán của mô hình học máy.

Trong nghiên cứu này, Mean Absolute SHAP được sử dụng để đánh giá tầm quan trọng tương đối của các feature trên bộ dữ liệu Digital Burnout quốc tế.

Đồng thời, thứ hạng SHAP cũng được xác định nhằm hỗ trợ đánh giá mức độ bằng chứng của từng indicator.

In [219]:
# Chuẩn hóa kết quả SHAP Feature Importance

shap_evidence = (
    shap_feature_importance.copy()
)

shap_evidence = (
    shap_evidence
    .sort_values(
        by="mean_abs_shap",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)

shap_evidence[
    "shap_rank"
] = (
    shap_evidence.index + 1
)

display(
    shap_evidence.head(10)
)

,feature,mean_shap,mean_abs_shap,shap_rank
0,emotional_exhaustion,-0.255703,0.761339,1
1,stress_level,-0.146301,0.565297,2
2,work_satisfaction,-0.059630,0.365784,3
3,daily_screen_time,-0.045927,0.293019,4
4,doomscrolling_duration,-0.036148,0.271082,5
5,late_night_device_usage,-0.020037,0.245732,6
6,distraction_frequency,-0.019225,0.218274,7
7,sleep_hours,-0.012258,0.169751,8
8,physical_activity,-0.008846,0.149915,9
9,notification_count,-0.006728,0.138617,10


## 3.5 Overall Evidence Assessment

Tổng hợp toàn bộ bằng chứng của từng Digital Burnout Indicator từ cơ sở lý thuyết, khả năng đo lường trên bộ dữ liệu Việt Nam và kết quả phân tích thực nghiệm trên bộ dữ liệu quốc tế.

Mỗi indicator sẽ được đánh giá theo bốn thành phần:

- Theoretical Evidence
- Vietnam Measurement Evidence
- International Validation Evidence
- Explainable AI Evidence

Dựa trên các thành phần này, indicator sẽ được phân loại thành ba mức độ bằng chứng:

- Strong Evidence
- Moderate Evidence
- Weak Evidence

In [220]:
# Khởi tạo DBI Indicator Evidence Matrix

dbi_indicator_evidence_matrix = (
    feature_indicator_mapping.copy()
)

display(
    dbi_indicator_evidence_matrix.head()
)

,feature,indicator_id,indicator_name,dbi_dimension,mapping_type
0,daily_screen_time,DE01,Daily Total Screen Time,Digital Exposure,Direct
1,social_media_hours,DE02,Media Multitasking Frequency,Digital Exposure,Proxy
2,app_switch_frequency,DE02,Media Multitasking Frequency,Digital Exposure,Direct
3,notification_count,DE04,Notification Check Frequency,Digital Exposure,Direct
4,smartphone_unlocks,DE04,Notification Check Frequency,Digital Exposure,Proxy


In [221]:
# Ghép thông tin lý thuyết

dbi_indicator_evidence_matrix = (

    dbi_indicator_evidence_matrix

    .merge(

        theoretical_evidence[
            [
                "indicator_id",
                "theory_supported",
                "theoretical_support"
            ]
        ],

        on="indicator_id",

        how="left"

    )

)

display(
    dbi_indicator_evidence_matrix.head()
)

,feature,indicator_id,indicator_name,dbi_dimension,mapping_type,theory_supported,theoretical_support
0,daily_screen_time,DE01,Daily Total Screen Time,Digital Exposure,Direct,True,Technostress Creators Framework (Techno-overload)
1,social_media_hours,DE02,Media Multitasking Frequency,Digital Exposure,Proxy,True,Media Multitasking Index (MMI)
2,app_switch_frequency,DE02,Media Multitasking Frequency,Digital Exposure,Direct,True,Media Multitasking Index (MMI)
3,notification_count,DE04,Notification Check Frequency,Digital Exposure,Direct,True,Technostress Creators - Techno-invasion
4,smartphone_unlocks,DE04,Notification Check Frequency,Digital Exposure,Proxy,True,Technostress Creators - Techno-invasion


In [222]:
# Ghép kết quả Feature Validation

dbi_indicator_evidence_matrix = (

    dbi_indicator_evidence_matrix

    .merge(

        international_validated_features[
            [
                "feature",
                "feature_status",
                "importance",
                "rf_rank"
            ]
        ],

        on="feature",

        how="left"

    )

)

display(
    dbi_indicator_evidence_matrix.head()
)

,feature,indicator_id,indicator_name,dbi_dimension,mapping_type,theory_supported,theoretical_support,feature_status,importance,rf_rank
0,daily_screen_time,DE01,Daily Total Screen Time,Digital Exposure,Direct,True,Technostress Creators Framework (Techno-overload),Core Feature,0.067815,3
1,social_media_hours,DE02,Media Multitasking Frequency,Digital Exposure,Proxy,True,Media Multitasking Index (MMI),Supporting Feature,0.022803,13
2,app_switch_frequency,DE02,Media Multitasking Frequency,Digital Exposure,Direct,True,Media Multitasking Index (MMI),Supporting Feature,0.025059,11
3,notification_count,DE04,Notification Check Frequency,Digital Exposure,Direct,True,Technostress Creators - Techno-invasion,Validated Feature,0.035040,8
4,smartphone_unlocks,DE04,Notification Check Frequency,Digital Exposure,Proxy,True,Technostress Creators - Techno-invasion,Supporting Feature,0.024923,12


In [223]:
# Ghép kết quả SHAP

dbi_indicator_evidence_matrix = (

    dbi_indicator_evidence_matrix

    .merge(

        shap_evidence[
            [
                "feature",
                "mean_abs_shap",
                "shap_rank"
            ]
        ],

        on="feature",

        how="left"

    )

)

display(
    dbi_indicator_evidence_matrix.head()
)

,feature,indicator_id,indicator_name,dbi_dimension,mapping_type,theory_supported,theoretical_support,feature_status,importance,rf_rank,mean_abs_shap,shap_rank
0,daily_screen_time,DE01,Daily Total Screen Time,Digital Exposure,Direct,True,Technostress Creators Framework (Techno-overload),Core Feature,0.067815,3,0.293019,4
1,social_media_hours,DE02,Media Multitasking Frequency,Digital Exposure,Proxy,True,Media Multitasking Index (MMI),Supporting Feature,0.022803,13,0.005106,14
2,app_switch_frequency,DE02,Media Multitasking Frequency,Digital Exposure,Direct,True,Media Multitasking Index (MMI),Supporting Feature,0.025059,11,0.005068,15
3,notification_count,DE04,Notification Check Frequency,Digital Exposure,Direct,True,Technostress Creators - Techno-invasion,Validated Feature,0.035040,8,0.138617,10
4,smartphone_unlocks,DE04,Notification Check Frequency,Digital Exposure,Proxy,True,Technostress Creators - Techno-invasion,Supporting Feature,0.024923,12,0.006017,12


In [224]:
# Hàm đánh giá mức độ bằng chứng

def classify_evidence(feature_status, shap_rank):

    if (
        feature_status == "Core Feature"
        and
        pd.notna(shap_rank)
        and
        shap_rank <= 10
    ):
        return "Strong"

    elif (
        feature_status == "Validated Feature"
        and
        pd.notna(shap_rank)
        and
        shap_rank <= 10
    ):
        return "Moderate"

    else:
        return "Weak"


dbi_indicator_evidence_matrix[
    "evidence_level"
] = (

    dbi_indicator_evidence_matrix.apply(

        lambda row:

        classify_evidence(

            row["feature_status"],

            row["shap_rank"]

        ),

        axis=1

    )

)

In [225]:
# Sắp xếp theo mức độ đóng góp của SHAP

evidence_columns = [

    "indicator_id",
    "indicator_name",
    "dbi_dimension",
    "feature",
    "feature_status",
    "rf_rank",
    "shap_rank",
    "mean_abs_shap",
    "evidence_level"

]

display(

    dbi_indicator_evidence_matrix[
        evidence_columns
    ]

    .sort_values(
        by="shap_rank"
    )

)

,indicator_id,indicator_name,dbi_dimension,feature,feature_status,rf_rank,shap_rank,mean_abs_shap,evidence_level
15,PS03,Emotional Exhaustion,Psychological Symptoms,emotional_exhaustion,Core Feature,1,1,0.761339,Strong
14,PS03,Emotional Exhaustion,Psychological Symptoms,stress_level,Core Feature,2,2,0.565297,Strong
0,DE01,Daily Total Screen Time,Digital Exposure,daily_screen_time,Core Feature,3,4,0.293019,Strong
6,PS02,Fear of Missing Out (FOMO),Psychological Symptoms,doomscrolling_duration,Core Feature,5,5,0.271082,Strong
5,DE05,Late-night Screen Exposure,Digital Exposure,late_night_device_usage,Validated Feature,17,6,0.245732,Moderate
7,CP03,Sustained Attention Decline,Cognitive Performance,distraction_frequency,Core Feature,6,7,0.218274,Strong
12,SR01,Sleep Onset Latency,Sleep & Recovery,sleep_hours,Validated Feature,7,8,0.169751,Moderate
3,DE04,Notification Check Frequency,Digital Exposure,notification_count,Validated Feature,8,10,0.138617,Moderate
10,CP02,Techno-Complexity/Cognitive Overload,Cognitive Performance,deep_work_hours,Validated Feature,10,11,0.077723,Weak
4,DE04,Notification Check Frequency,Digital Exposure,smartphone_unlocks,Supporting Feature,12,12,0.006017,Weak


In [226]:
indicator_weight_table = (

    dbi_indicator_evidence_matrix

    .groupby(

        [
            "indicator_id",
            "indicator_name",
            "dbi_dimension"
        ]

    )

    .agg(

        feature_count=(
            "feature",
            "count"
        ),

        mean_importance=(
            "importance",
            "mean"
        ),
        
        mean_shap=(
            "mean_abs_shap",
            "mean"
        ),

        evidence_strength=(
            "evidence_level",
            "first"
        )

    )
    .reset_index()
)

# 4. Indicator Evidence Aggregation

Sau khi xác định được mối quan hệ giữa các feature và Digital Burnout Indicator, bước tiếp theo là tổng hợp các bằng chứng thực nghiệm ở cấp độ indicator.

Trong nghiên cứu này, một Digital Burnout Indicator có thể được đo lường thông qua một hoặc nhiều feature đã được xác thực từ bộ dữ liệu quốc tế. Vì vậy, cần tổng hợp toàn bộ thông tin của các feature thành bằng chứng đại diện cho từng indicator.

Quá trình tổng hợp bao gồm:

- Tổng hợp các feature thuộc cùng một indicator.
- Đánh giá mức độ hỗ trợ của indicator dựa trên Feature Validation và Explainable AI.
- Phân loại loại hình đo lường của indicator.
- Xây dựng bảng Digital Burnout Indicator Evidence.

Kết quả của phần này không phải là trọng số của indicator mà là cơ sở thực nghiệm phục vụ việc xây dựng Digital Burnout Index trong Notebook 02.

## 4.1 Aggregate Feature Evidence

Tổng hợp các feature đã được xác thực thành từng Digital Burnout Indicator.

In [227]:
indicator_evidence_summary = (

    dbi_indicator_evidence_matrix

    .groupby(
        [
            "indicator_id",
            "indicator_name",
            "dbi_dimension"
        ],
        as_index=False
    )

    .agg(

        mapped_features=(
            "feature",
            lambda x: ", ".join(sorted(x))
        ),

        mapped_feature_count=(
            "feature",
            "count"
        ),

        rf_importance=(
            "importance",
            "mean"
        ),

        shap_importance=(
            "mean_abs_shap",
            "mean"
        ),

        feature_status=(
            "feature_status",
            list
        )

    )

)

## 4.2 Indicator Evidence Assessment

Đánh giá mức độ bằng chứng của từng Digital Burnout Indicator dựa trên chất lượng của các feature đại diện.

In [228]:
def determine_evidence(status_list):

    if "Core Feature" in status_list:
        return "Strong"

    elif "Validated Feature" in status_list:
        return "Moderate"

    else:
        return "Weak"


indicator_evidence_summary["evidence_strength"] = (

    indicator_evidence_summary["feature_status"]

    .apply(determine_evidence)

)

indicator_evidence_summary.drop(

    columns="feature_status",

    inplace=True

)

## 4.3 Indicator Measurement Profile

Xác định đặc điểm đo lường của từng Digital Burnout Indicator.

In [229]:
indicator_evidence_summary["measurement_type"] = np.where(

    indicator_evidence_summary["mapped_feature_count"] == 1,

    "Single-feature",

    "Composite"

)

indicator_evidence_summary["framework_status"] = "Retained"

## 4.4 Indicator Evidence Summary

Tổng hợp toàn bộ bằng chứng thực nghiệm của Digital Burnout Indicator Framework.

In [230]:
strength_order={

    "Strong":3,

    "Moderate":2,

    "Weak":1

}

indicator_evidence_summary["strength_order"]=(
    indicator_evidence_summary["evidence_strength"]
    .map(strength_order)
)

In [231]:
indicator_evidence_summary=(

    indicator_evidence_summary

    .sort_values(

        by=[

            "strength_order",

            "shap_importance",

            "rf_importance"

        ],

        ascending=False

    )

    .drop(columns="strength_order")

    .reset_index(drop=True)

)

In [232]:
display(indicator_evidence_summary)

,indicator_id,indicator_name,dbi_dimension,mapped_features,mapped_feature_count,rf_importance,shap_importance,evidence_strength,measurement_type,framework_status
0,PS03,Emotional Exhaustion,Psychological Symptoms,"emotional_exhaustion, stress_level",2,0.181337,0.663318,Strong,Composite,Retained
1,DE01,Daily Total Screen Time,Digital Exposure,daily_screen_time,1,0.067815,0.293019,Strong,Single-feature,Retained
2,PS02,Fear of Missing Out (FOMO),Psychological Symptoms,doomscrolling_duration,1,0.054545,0.271082,Strong,Single-feature,Retained
3,CP03,Sustained Attention Decline,Cognitive Performance,"concentration_score, distraction_frequency, fo...",3,0.024205,0.074777,Strong,Composite,Retained
4,DE05,Late-night Screen Exposure,Digital Exposure,late_night_device_usage,1,0.020974,0.245732,Moderate,Single-feature,Retained
5,SR01,Sleep Onset Latency,Sleep & Recovery,sleep_hours,1,0.037063,0.169751,Moderate,Single-feature,Retained
6,CP02,Techno-Complexity/Cognitive Overload,Cognitive Performance,deep_work_hours,1,0.025811,0.077723,Moderate,Single-feature,Retained
7,DE04,Notification Check Frequency,Digital Exposure,"notification_count, smartphone_unlocks",2,0.029981,0.072317,Moderate,Composite,Retained
8,CP04,Cross-Domain Interference Susceptibility,Cognitive Performance,task_completion_rate,1,0.022150,0.005185,Weak,Single-feature,Retained
9,DE02,Media Multitasking Frequency,Digital Exposure,"app_switch_frequency, social_media_hours",2,0.023931,0.005087,Weak,Composite,Retained


Kết quả của Notebook 01 là Digital Burnout Indicator Framework, bao gồm 12 indicator được xây dựng từ 17 feature đã được xác thực trên bộ dữ liệu quốc tế. Mỗi indicator được mô tả bởi nhóm feature đại diện, mức độ hỗ trợ thực nghiệm từ Feature Validation và Explainable AI, cùng loại hình đo lường (Single-feature hoặc Composite). Framework này đóng vai trò là nền tảng cho việc xác định trọng số và xây dựng công thức tính Digital Burnout Index trong Notebook 02.

# 5. Export Final DBI Framework

Xuất bộ Digital Burnout Indicator Framework đã được xây dựng từ cơ sở lý thuyết và bằng chứng thực nghiệm.

Framework này sẽ được sử dụng làm đầu vào cho Notebook 02 để xác định trọng số indicator và xây dựng công thức tính Digital Burnout Index (DBI).

In [233]:
dbi_framework = (

    indicator_evidence_summary

    .copy()

)

In [234]:
display(dbi_framework)

print(dbi_framework.shape)

,indicator_id,indicator_name,dbi_dimension,mapped_features,mapped_feature_count,rf_importance,shap_importance,evidence_strength,measurement_type,framework_status
0,PS03,Emotional Exhaustion,Psychological Symptoms,"emotional_exhaustion, stress_level",2,0.181337,0.663318,Strong,Composite,Retained
1,DE01,Daily Total Screen Time,Digital Exposure,daily_screen_time,1,0.067815,0.293019,Strong,Single-feature,Retained
2,PS02,Fear of Missing Out (FOMO),Psychological Symptoms,doomscrolling_duration,1,0.054545,0.271082,Strong,Single-feature,Retained
3,CP03,Sustained Attention Decline,Cognitive Performance,"concentration_score, distraction_frequency, fo...",3,0.024205,0.074777,Strong,Composite,Retained
4,DE05,Late-night Screen Exposure,Digital Exposure,late_night_device_usage,1,0.020974,0.245732,Moderate,Single-feature,Retained
5,SR01,Sleep Onset Latency,Sleep & Recovery,sleep_hours,1,0.037063,0.169751,Moderate,Single-feature,Retained
6,CP02,Techno-Complexity/Cognitive Overload,Cognitive Performance,deep_work_hours,1,0.025811,0.077723,Moderate,Single-feature,Retained
7,DE04,Notification Check Frequency,Digital Exposure,"notification_count, smartphone_unlocks",2,0.029981,0.072317,Moderate,Composite,Retained
8,CP04,Cross-Domain Interference Susceptibility,Cognitive Performance,task_completion_rate,1,0.022150,0.005185,Weak,Single-feature,Retained
9,DE02,Media Multitasking Frequency,Digital Exposure,"app_switch_frequency, social_media_hours",2,0.023931,0.005087,Weak,Composite,Retained


(12, 10)


In [235]:
output_path = (
    "../../data/processed/cross_dataset_analysis/dbi_framework.csv"
)

dbi_framework.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"Saved to: {output_path}")

Saved to: ../../data/processed/cross_dataset_analysis/dbi_framework.csv


## 6. Xây dựng trọng số Indicator

Xác định mức độ đóng góp của từng Indicator trong Digital Burnout Framework để phục vụ quá trình tính toán điểm DBI.

Trọng số Indicator được xây dựng dựa trên ba nguồn bằng chứng:

- SHAP importance từ mô hình học máy.
- Random Forest importance từ quá trình Feature Validation.
- Evidence Strength từ cơ sở lý thuyết.

Trọng số cuối cùng được chuẩn hóa để tổng các Indicator Weight bằng 1.


In [236]:
# Gán điểm bằng chứng lý thuyết

evidence_score_mapping = {
    "Strong": 1.0,
    "Moderate": 0.7,
    "Weak": 0.4
}


dbi_framework_weighted = dbi_framework.copy()


dbi_framework_weighted["evidence_score"] = (
    dbi_framework_weighted["evidence_strength"]
    .map(evidence_score_mapping)
)


# Tính trọng số thô của Indicator

dbi_framework_weighted["raw_weight"] = (
    0.5 * dbi_framework_weighted["shap_importance"]
    +
    0.3 * dbi_framework_weighted["rf_importance"]
    +
    0.2 * dbi_framework_weighted["evidence_score"]
)


# Chuẩn hóa trọng số

dbi_framework_weighted["indicator_weight"] = (
    dbi_framework_weighted["raw_weight"]
    /
    dbi_framework_weighted["raw_weight"].sum()
)


print("Đã tính trọng số cho các Indicator.")

Đã tính trọng số cho các Indicator.


In [237]:
dbi_framework_weighted[
    [
        "indicator_id",
        "indicator_name",
        "indicator_weight"
    ]
]

,indicator_id,indicator_name,indicator_weight
0,PS03,Emotional Exhaustion,0.211084
1,DE01,Daily Total Screen Time,0.132131
2,PS02,Fear of Missing Out (FOMO),0.126747
3,CP03,Sustained Attention Decline,0.088117
4,DE05,Late-night Screen Exposure,0.096944
5,SR01,Sleep Onset Latency,0.084999
6,CP02,Techno-Complexity/Cognitive Overload,0.067210
7,DE04,Notification Check Frequency,0.066687
8,CP04,Cross-Domain Interference Susceptibility,0.032141
9,DE02,Media Multitasking Frequency,0.032316


In [238]:
# Xuất DBI Framework có trọng số

dbi_framework_weighted.to_csv(
    "../../data/processed/cross_dataset_analysis/dbi_framework.csv",
    index=False,
    encoding="utf-8"
)


print("Đã xuất DBI Framework có trọng số.")

Đã xuất DBI Framework có trọng số.
